In [1]:
from dotenv import load_dotenv
import os
from football_analytics.utils import supabase as db
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
import football_analytics.data_processing.preprocessing as preprocessing
from importlib import reload
import json
reload(preprocessing)

<module 'football_analytics.data_processing.preprocessing' from 'C:\\Users\\david\\Documents\\Git\\football-analytics\\src\\football_analytics\\data_processing\\preprocessing.py'>

## Loading Database Info

In [2]:
# Load variables from .env into the environment
load_dotenv()

backend = db.get_db_backend()
print(f"DB backend: {backend}")


DB backend: postgres


In [28]:
# DB client handled by football_analytics.utils.supabase


## Process Statsbomb Data

#### Shots

In [3]:
table_name = "shots"
response = db.fetch_rows(table_name=table_name, columns="*", limit=1)
print(response)

[RealDictRow([('created_at', datetime.datetime(2025, 12, 8, 21, 4, 18, 607957, tzinfo=datetime.timezone.utc)), ('statsbomb_event_id', '00006e24-97ca-455a-b9e1-ce12770a34c7'), ('x1', Decimal('98.7')), ('y1', Decimal('59.4')), ('distance_to_goal', Decimal('28.81058833137567')), ('angle_to_goal_deg', Decimal('11.82266443477366')), ('keeper_distance', Decimal('22.663186007267374')), ('min_defender_distance', Decimal('9.656603957913976')), ('avg_defender_distance', Decimal('12.041986803377831')), ('num_def_in_shot_triangle', Decimal('0')), ('num_teammates_in_box', Decimal('0')), ('shot_to_min_def_ratio', Decimal('2.9804250238046275')), ('shot_type', 'Open Play'), ('body_part', 'Right Foot'), ('outcome', 'Off T'), ('full_json', {'id': '00006e24-97ca-455a-b9e1-ce12770a34c7', 'shot': {'type': {'id': 87, 'name': 'Open Play'}, 'outcome': {'id': 98, 'name': 'Off T'}, 'body_part': {'id': 40, 'name': 'Right Foot'}, 'technique': {'id': 92, 'name': 'Lob'}, 'first_time': True, 'end_location': [120.0, 

In [31]:
folder_path = Path("../../data/statsbomb/open-data-master/data/events")
json_files = list(folder_path.glob("*.json"))

table_name = "shots"
BATCHSIZE_SHOT_UPSERT = 1000  # tune to 500–2000 depending on timeouts
MATCH_LIMIT = 1e6
skip_first_n_matches = -1

shots_to_upsert = []

for match_index, json_file in tqdm(enumerate(json_files), desc="Processing...", total=len(json_files)):
    if match_index < skip_first_n_matches:
        continue
    if match_index > MATCH_LIMIT:
        break

    match_id = json_file.stem

    # Load events without pandas
    events = json.loads(json_file.read_bytes())
    shot_events = [e for e in events if e.get("shot")]

    # Extract features
    for event in shot_events:
        shot_data = preprocessing.extract_shot_features(event, match_id)
        shots_to_upsert.append(shot_data)

    # Upsert in larger batches to reduce overhead
    if len(shots_to_upsert) >= BATCHSIZE_SHOT_UPSERT:
        # returning="minimal" avoids row payloads (if your client supports it)
        db.upsert_rows(table_name, shots_to_upsert, conflict_columns="statsbomb_event_id")
        shots_to_upsert = []

# Flush remainder
if shots_to_upsert:
    db.upsert_rows(table_name, shots_to_upsert, conflict_columns="statsbomb_event_id")
    shots_to_upsert = []

Processing...:   0%|          | 0/3464 [00:00<?, ?it/s]

#### Competion

In [19]:
json_file = "../../data/statsbomb/open-data-master/data/competitions.json"
with open(json_file, 'r') as f:
    competitions_to_upsert = json.load(f)

In [20]:
table_name = "competitions"
response = db.upsert_rows(table_name, competitions_to_upsert, conflict_columns="competition_id, season_id")

#### Teams

In [22]:
table_name = "teams"
matches_folder = Path("../../data/statsbomb/open-data-master/data/matches")
match_files = list(matches_folder.glob("*/*.json"))
teams_by_id = {}

for match_file in tqdm(match_files, desc="Collecting teams", total=len(match_files)):
    matches = json.loads(match_file.read_bytes())
    for match in matches:
        for side in ("home_team", "away_team"):
            team_row = preprocessing.extract_team_row(match.get(side))
            if team_row and team_row.get("team_id"):
                teams_by_id[team_row["team_id"]] = team_row

teams_df = pd.DataFrame(teams_by_id.values())
# Replace NaN/NaT/inf with None to keep JSON serializable
teams_df = teams_df.replace([pd.NA, pd.NaT, float("inf"), float("-inf")], None)
teams_df = teams_df.where(pd.notnull(teams_df), None)
teams_to_upsert = []
for row in teams_df.to_dict(orient="records"):
    clean_row = {}
    for k, v in row.items():
        if pd.isna(v):
            clean_row[k] = None
            continue
        if isinstance(v, (float, int)) and math.isinf(v):
            clean_row[k] = None
            continue
        clean_row[k] = v
    teams_to_upsert.append(clean_row)



In [27]:
response = db.upsert_rows(table_name, teams_to_upsert, conflict_columns="team_id")
print(f"Prepared {len(teams_to_upsert)} teams for upsert")


Prepared 312 teams for upsert


#### Matches


In [25]:
id_fields = ['away_team_id', 'competition_id', 'competition_stage_id', 'home_team_id', 'match_id', 'referee_id', 'season_id', 'stadium_id']
table_name = "matches"
import math
matches_folder = Path("../../data/statsbomb/open-data-master/data/matches")
match_files = list(matches_folder.glob("*/*.json"))
matches_by_id = {}

for match_file in tqdm(match_files, desc="Collecting matches", total=len(match_files)):
    matches = json.loads(match_file.read_bytes())
    for match in matches:
        match_row = preprocessing.extract_match_row(match)
        if match_row and match_row.get("match_id"):
            matches_by_id[match_row["match_id"]] = match_row

matches_df = pd.DataFrame(matches_by_id.values())
int_fields = id_fields + ["home_score", "away_score", "match_week"]
for col in int_fields:
    if col in matches_df.columns:
        matches_df[col] = matches_df[col].astype("Int64")
# Replace NaN/NaT/inf with None to keep JSON serializable
matches_df = matches_df.replace([pd.NA, pd.NaT, float("inf"), float("-inf")], None)
matches_df = matches_df.where(pd.notnull(matches_df), None)
matches_to_upsert = []
for row in matches_df.to_dict(orient="records"):
    clean_row = {}
    for k, v in row.items():
        if k in id_fields and isinstance(v, float) and v.is_integer():
            v = int(v)
        if pd.isna(v):
            clean_row[k] = None
            continue
        if isinstance(v, (float, int)) and math.isinf(v):
            clean_row[k] = None
            continue
        clean_row[k] = v
    matches_to_upsert.append(clean_row)



In [26]:
matches_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3464 entries, 0 to 3463
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   match_id              3464 non-null   Int64 
 1   match_date            3464 non-null   object
 2   kick_off              3459 non-null   object
 3   competition_id        3464 non-null   Int64 
 4   season_id             3464 non-null   Int64 
 5   home_team_id          3464 non-null   Int64 
 6   away_team_id          3464 non-null   Int64 
 7   home_score            3464 non-null   Int64 
 8   away_score            3464 non-null   Int64 
 9   match_status          3464 non-null   object
 10  match_status_360      3464 non-null   object
 11  last_updated          3464 non-null   object
 12  last_updated_360      1828 non-null   object
 13  match_week            3464 non-null   Int64 
 14  competition_stage_id  3464 non-null   Int64 
 15  stadium_id            3454 non-null   

In [27]:
df = pd.DataFrame(matches_to_upsert)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3464 entries, 0 to 3463
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   match_id              3464 non-null   int64  
 1   match_date            3464 non-null   object 
 2   kick_off              3459 non-null   object 
 3   competition_id        3464 non-null   int64  
 4   season_id             3464 non-null   int64  
 5   home_team_id          3464 non-null   int64  
 6   away_team_id          3464 non-null   int64  
 7   home_score            3464 non-null   int64  
 8   away_score            3464 non-null   int64  
 9   match_status          3464 non-null   object 
 10  match_status_360      3464 non-null   object 
 11  last_updated          3464 non-null   object 
 12  last_updated_360      1828 non-null   object 
 13  match_week            3464 non-null   int64  
 14  competition_stage_id  3464 non-null   int64  
 15  stadium_id           

In [28]:
if matches_to_upsert:
    response = db.upsert_rows(table_name, matches_to_upsert, conflict_columns="match_id")
else:
    print("No matches collected")
print(f"Prepared {len(matches_to_upsert)} matches for upsert")


Prepared 3464 matches for upsert


#### Players

In [ ]:
table_name = "players"
import math
events_folder = Path("../../data/statsbomb/open-data-master/data/events")
event_files = list(events_folder.glob("*.json"))
players_by_id = {}

for event_file in tqdm(event_files, desc="Collecting players", total=len(event_files)):
    events = json.loads(event_file.read_bytes())
    for ev in events:
        player_row = preprocessing.extract_player_row(ev)
        if player_row and player_row.get("statsbomb_player_id"):
            players_by_id[player_row["statsbomb_player_id"]] = player_row

players_df = pd.DataFrame(players_by_id.values())
# Replace NaN/NaT/inf with None to keep JSON serializable
players_df = players_df.replace([pd.NA, pd.NaT, float("inf"), float("-inf")], None)
players_df = players_df.where(pd.notnull(players_df), None)



In [17]:
players_to_upsert = []
for row in players_df.to_dict(orient="records"):
    clean_row = {}
    for k, v in row.items():
        if pd.isna(v):
            clean_row[k] = None
            continue
        if isinstance(v, (float, int)) and math.isinf(v):
            clean_row[k] = None
            continue
        clean_row[k] = v
    players_to_upsert.append(clean_row)



In [19]:
if players_to_upsert:
    response = db.upsert_rows(table_name, players_to_upsert, conflict_columns="statsbomb_player_id")
else:
    print("No players collected")
print(f"Prepared {len(players_to_upsert)} players for upsert")


Prepared 9043 players for upsert


#### All Events